# Part 3: ML Model Training & Product-Level Discount System

## 🎯 Learning Objectives
- Train sentiment analysis model using TF-IDF + Logistic Regression
- Evaluate model performance
- Create smart discount system using:
  - Product sentiment (% negative reviews)
  - Stock levels (overstocked = higher discount)
  - Sales velocity (slow sales = higher discount)
- Analyze business impact

## 🏢 Business Logic

**Individual Review** → Sentiment Score

**Product Aggregation** →
- % Negative reviews
- Average sentiment confidence
- Stock level
- Sales velocity

**Discount Formula** →
```
Discount = f(sentiment_score, stock_pressure, sales_velocity)
```

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

print("✅ Libraries loaded")

In [ ]:
# Workshop Progress Tracker
notebooks = ["01 Extract", "02 Prepare", "03 Storage", "04 ML", "05 Deploy"]
current = 3  # This is notebook 04

print("="*70)
print("📊 WORKSHOP PROGRESS")
print("="*70)
for i, nb in enumerate(notebooks):
    if i < current:
        print(f"✅ {nb}")
    elif i == current:
        print(f"👉 {nb} ← YOU ARE HERE")
    else:
        print(f"⬜ {nb}")
print("="*70)

## Step 1: Load Gold Dataset

We'll load the final enriched dataset from the Gold layer (created in Notebook 2).

In [ ]:
# Load Gold dataset (enriched with stock and sales data)
df = pd.read_csv('../data/gold/final_dataset.csv')

print("📊 Gold Dataset Loaded")
print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## Step 2: Split Data (Train/Test)

**Why split the data?**
- **Training set (80%)**: Used to train the model
- **Test set (20%)**: Used to evaluate performance on unseen data
- Prevents overfitting and gives realistic performance estimates

In [ ]:
# Features (X) and target (y)
X = df['review_text_clean']
y = df['sentiment']

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} reviews")
print(f"Test set: {len(X_test)} reviews")
print(f"\nTraining set sentiment distribution:")
print(y_train.value_counts())

## Step 3: Text Vectorization (TF-IDF)

**What is TF-IDF?**
- **TF (Term Frequency)**: How often a word appears in a document
- **IDF (Inverse Document Frequency)**: How unique/important a word is across all documents
- Converts text into numerical features for ML

Example:
- "best" appears often in positive reviews → high TF-IDF for positive sentiment
- "worst" appears often in negative reviews → high TF-IDF for negative sentiment
- Common words like "the", "is" → low TF-IDF (not discriminative)

In [ ]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=500,  # Use top 500 most important words
    min_df=2,  # Word must appear in at least 2 documents
    max_df=0.9,  # Ignore words that appear in >90% of documents
    ngram_range=(1, 2)  # Use single words and two-word phrases
)

# Fit on training data and transform both train and test
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF feature shape: {X_train_tfidf.shape}")
print(f"Number of features (words/phrases): {len(vectorizer.get_feature_names_out())}")
print(f"\nTop 20 features:")
print(vectorizer.get_feature_names_out()[:20])

## Step 4: Train Sentiment Analysis Model

We'll use **Logistic Regression** - a simple but effective classifier for text.

**How it works:**
- Learns weights for each word/phrase
- Positive words ("best", "fresh") → positive weight
- Negative words ("worst", "stale") → negative weight
- Combines weights to predict sentiment

In [ ]:
# Train Logistic Regression classifier
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

print("Training model...")
model.fit(X_train_tfidf, y_train)
print("Model trained successfully!")

## Step 5: Evaluate Model Performance

### Analyze Output Model
How well does our sentiment classifier perform?

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")
print("\n" + "="*60)
print("Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['negative', 'neutral', 'positive'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix - Sentiment Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Interpretation:")
print("- Diagonal values (correct predictions): Higher is better")
print("- Off-diagonal values (mistakes): Lower is better")

## Step 6: Analyze "Why" - Most Important Words

Let's see which words are most predictive for each sentiment.

In [ ]:
# Get feature names
feature_names = vectorizer.get_feature_names_out()

# Get class labels and coefficients
classes = model.classes_
coefficients = model.coef_

# Show top words for each sentiment
n_top = 10

for i, sentiment in enumerate(classes):
    print(f"\n{'='*60}")
    print(f"Top {n_top} words for '{sentiment.upper()}' sentiment:")
    print(f"{'='*60}")
    
    # Get indices of top features
    top_indices = coefficients[i].argsort()[-n_top:][::-1]
    
    for idx in top_indices:
        word = feature_names[idx]
        weight = coefficients[i][idx]
        print(f"  {word:20s} (weight: {weight:+.3f})")

### 🔍 OPTIONAL CHALLENGE: Can you fool the model?
Task: Write a review that SOUNDS positive but gets classified as negative or vice versa

Example: 'This spinach is not bad' ← sounds neutral but might confuse model

💡 Bonus points if you can explain WHY the model got confused!
Try it in the prediction cell above


## Step 7: Make Predictions on New Reviews

Let's test our model with some example reviews.

In [ ]:
def predict_sentiment(review_text):
    """
    Predict sentiment for a review text
    """
    # Preprocess text (same as training)
    import re
    text_clean = review_text.lower()
    text_clean = ' '.join(text_clean.split())
    text_clean = re.sub(r'[^a-z\s.,!?]', '', text_clean)
    
    # Vectorize
    text_tfidf = vectorizer.transform([text_clean])
    
    # Predict
    sentiment = model.predict(text_tfidf)[0]
    probabilities = model.predict_proba(text_tfidf)[0]
    
    return sentiment, probabilities

# Test with example reviews
test_reviews = [
    "This is the worst spinach ever! Not fresh at all.",
    "Best potatoes in my life! Super fresh and delicious.",
    "The carrot was a bit stale and disappointing.",
    "These tomatoes are okay, nothing special."
]

print("Test Predictions:")
print("="*80)

for review in test_reviews:
    sentiment, probs = predict_sentiment(review)
    print(f"\nReview: {review}")
    print(f"Predicted Sentiment: {sentiment.upper()}")
    print(f"Confidence: {probs.max():.1%}")
    print(f"Probabilities: Negative={probs[0]:.2%}, Neutral={probs[1]:.2%}, Positive={probs[2]:.2%}")

**💡 Group Challenge:** Who can write a review that gets a 90% discount? Try it and discuss with your neighbor - why did certain words trigger such a high discount?

### 🎮 YOUR TURN: Interactive Prediction Station!

Now it's your turn to test the model! Try typing your own product reviews and see what happens.

## Step 8: Discount Recommendation System

Now let's create the discount recommendation logic!

### Discount Rules:
- **Negative sentiment (1-2 stars)**: 50-90% discount
- **Neutral sentiment (3 stars)**: 20-30% discount
- **Positive sentiment (4-5 stars)**: 0-10% discount

We'll also consider:
- Confidence of the prediction (higher confidence → more extreme discount)
- Sales volume (lower sales + negative sentiment → higher discount)

**💬 Group Discussion (2 minutes):**
- Which product would you focus on first as a store manager?
- Why might certain products have more negative reviews? (Think: perishability, freshness, storage)

In [ ]:
# Create a fun leaderboard
print("="*70)
print("🏆 PRODUCT SENTIMENT LEADERBOARD 🏆")
print("="*70)

# Calculate sentiment scores (weighted)
product_stats_lb = df.groupby('product_name').agg({
    'sentiment': [
        ('negative_pct', lambda x: (x == 'negative').mean() * 100),
        ('positive_pct', lambda x: (x == 'positive').mean() * 100),
        ('total_reviews', 'count')
    ],
    'rating': 'mean'
}).round(1)

# Flatten columns
product_stats_lb.columns = ['_'.join(col) for col in product_stats_lb.columns]
product_stats_lb = product_stats_lb.reset_index()

# Create overall score (0-100, higher is better)
product_stats_lb['happiness_score'] = (
    product_stats_lb['sentiment_positive_pct'] -
    product_stats_lb['sentiment_negative_pct']
)

# Sort and rank
product_stats_lb = product_stats_lb.sort_values('happiness_score', ascending=False)
product_stats_lb['rank'] = range(1, len(product_stats_lb) + 1)

# Display with fun formatting
print("\n🥇 HAPPIEST PRODUCTS (Customer Loves Them)")
print("-"*70)
for _, row in product_stats_lb.head(3).iterrows():
    emoji = "🥇" if row['rank'] == 1 else "🥈" if row['rank'] == 2 else "🥉"
    print(f"{emoji} #{int(row['rank'])} {row['product_name']:15} | "
          f"Happiness Score: {row['happiness_score']:+.1f} | "
          f"⭐ {row['rating_mean']:.1f}/5.0")

print("\n\n😰 NEEDS ATTENTION (High Negative Sentiment)")
print("-"*70)
for _, row in product_stats_lb.tail(3).iterrows():
    print(f"⚠️  #{int(row['rank'])} {row['product_name']:15} | "
          f"Happiness Score: {row['happiness_score']:+.1f} | "
          f"❌ {row['sentiment_negative_pct']:.1f}% negative")

print("\n💡 BUSINESS INSIGHT: Which products need discount campaigns?")

## Step 9: Smart Discount Calculator (Using ALL Data!)


Now let's create a **smart discount system** that uses:
1. **Sentiment** (from ML model)
2. **Stock levels** (from inventory data) 
3. **Sales velocity** (from sales data)

### Business Logic:
```
Base Discount (from sentiment):
  - Negative: 50-80%
  - Neutral:  20-30%
  - Positive: 0-10%

Adjustments:
  + High stock / Low sales → +10-20% (need to clear inventory)
  + Low stock / High sales → -5-10% (product is popular)
```

In [ ]:
def calculate_smart_discount(sentiment, confidence, stock_level=None, sales_velocity=None):
    """
    Calculate discount based on sentiment AND business metrics
    
    Args:
        sentiment: 'positive', 'neutral', or 'negative'
        confidence: 0.0 to 1.0 (model confidence)
        stock_level: Units in stock (optional)
        sales_velocity: Avg daily sales (optional)
    
    Returns:
        discount: 0.0 to 1.0 (percentage as decimal)
    """
    
    # Step 1: Base discount from sentiment
    if sentiment == 'negative':
        base_discount = 0.50 + (confidence * 0.30)  # 50-80%
    elif sentiment == 'neutral':
        base_discount = 0.20 + (confidence * 0.10)  # 20-30%
    else:  # positive
        base_discount = 0.00 + (confidence * 0.10)  # 0-10%
    
    # Step 2: Adjust based on inventory pressure (if data available)
    inventory_adjustment = 0.0
    
    if stock_level is not None and sales_velocity is not None:
        # Calculate days of inventory
        days_of_inventory = stock_level / (sales_velocity + 0.1)  # +0.1 to avoid division by zero
        
        if days_of_inventory > 30:  # Overstocked (>30 days)
            inventory_adjustment = +0.20  # Add 20% discount
        elif days_of_inventory > 14:  # High stock (>2 weeks)
            inventory_adjustment = +0.10  # Add 10% discount
        elif days_of_inventory < 3:  # Low stock (<3 days)
            inventory_adjustment = -0.10  # Reduce discount by 10%
        elif days_of_inventory < 7:  # Medium-low stock
            inventory_adjustment = -0.05  # Reduce discount by 5%
    
    # Step 3: Combine and cap at 0-90%
    final_discount = base_discount + inventory_adjustment
    final_discount = max(0.0, min(0.90, final_discount))  # Cap between 0% and 90%
    
    return final_discount

print("Smart discount calculator ready!")
print("\nExample calculations:")
print("="*70)

# Test different scenarios
scenarios = [
    ('negative', 0.95, 100, 2, 'Bad reviews + Overstocked'),
    ('negative', 0.95, 10, 15, 'Bad reviews + Selling fast'),
    ('positive', 0.90, 5, 20, 'Good reviews + High demand'),
    ('neutral', 0.75, 50, 3, 'Mixed reviews + Slow sales'),
]

for sentiment, conf, stock, sales, description in scenarios:
    discount = calculate_smart_discount(sentiment, conf, stock, sales)
    print(f"{description}:")
    print(f"  Sentiment: {sentiment} ({conf:.0%} confidence)")
    print(f"  Stock: {stock} units | Sales: {sales}/day | Days left: {stock/sales:.1f}")
    print(f"  -> Recommended Discount: {discount:.0%}")
    print()

### Apply Smart Discounts to Real Products

Let's calculate discounts for actual products using their sentiment + inventory data!

In [ ]:
# Aggregate product-level data with sentiment + inventory + sales
product_discounts = df.groupby('product_name').agg({
    'sentiment': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'neutral',  # Most common sentiment
    'rating': 'mean',
    'avg_stock_level': 'mean',
    'avg_daily_sales': 'mean'
}).reset_index()

# Calculate confidence (using sentiment consistency as proxy)
sentiment_confidence = df.groupby('product_name')['sentiment'].apply(
    lambda x: (x == x.mode()[0]).mean() if len(x.mode()) > 0 else 0.5
).reset_index()
sentiment_confidence.columns = ['product_name', 'confidence']

product_discounts = product_discounts.merge(sentiment_confidence, on='product_name')

# Apply smart discount calculator
product_discounts['recommended_discount'] = product_discounts.apply(
    lambda row: calculate_smart_discount(
        row['sentiment'], 
        row['confidence'],
        row['avg_stock_level'],
        row['avg_daily_sales']
    ),
    axis=1
)

# Calculate inventory metrics
product_discounts['days_of_inventory'] = (
    product_discounts['avg_stock_level'] / 
    (product_discounts['avg_daily_sales'] + 0.1)
)

# Sort by discount (highest first)
product_discounts = product_discounts.sort_values('recommended_discount', ascending=False)

# Display results
print("="*70)
print("SMART DISCOUNT RECOMMENDATIONS")
print("="*70)
print("\nProducts ranked by recommended discount:\n")

for idx, row in product_discounts.iterrows():
    print(f"{row['product_name']:15} | Sentiment: {row['sentiment']:8} | "
          f"Stock: {row['avg_stock_level']:5.0f} | Sales: {row['avg_daily_sales']:4.1f}/day | "
          f"Days left: {row['days_of_inventory']:4.1f} | "
          f"Discount: {row['recommended_discount']:5.0%}")

print("\n" + "="*70)
print("\nKey Insights:")
print(f"- Highest discount: {product_discounts.iloc[0]['product_name']} ({product_discounts.iloc[0]['recommended_discount']:.0%})")
print(f"- Lowest discount: {product_discounts.iloc[-1]['product_name']} ({product_discounts.iloc[-1]['recommended_discount']:.0%})")
print(f"- Average discount: {product_discounts['recommended_discount'].mean():.0%}")

---

# Part D: MLOps Basics - Model Versioning & Validation

## 🎯 Before Deployment, We Need:
1. **Model Versioning** - Track which model is in production
2. **Model Registry** - Store model metadata
3. **Validation Checks** - Ensure model quality before deployment


## Step 1: Save Model with Versioning

Instead of just saving the model, we save it with **version metadata**.

In [ ]:
import json
from datetime import datetime

# Create version metadata
model_version = {
    "version": "1.0.0",
    "timestamp": datetime.now().isoformat(),
    "accuracy": float(accuracy),
    "precision": float(precision_score(y_test, y_pred, average='weighted')),
    "recall": float(recall_score(y_test, y_pred, average='weighted')),
    "f1_score": float(f1_score(y_test, y_pred, average='weighted')),
    "hyperparameters": {
        "C": 1.0,
        "max_iter": 100,
        "solver": "lbfgs"
    },
    "training_samples": len(X_train),
    "test_samples": len(X_test)
}

# Save model with version in filename
model_filename = f'sentiment_model_v{model_version["version"]}.pkl'
with open(f'../models/{model_filename}', 'wb') as f:
    pickle.dump(model, f)

# Save metadata as JSON
metadata_filename = f'sentiment_model_v{model_version["version"]}_metadata.json'
with open(f'../models/{metadata_filename}', 'w') as f:
    json.dump(model_version, f, indent=2)

# Save vectorizer (THIS WAS MISSING!)
vectorizer_filename = f'tfidf_vectorizer.pkl'
with open(f'../models/{vectorizer_filename}', 'wb') as f:
    pickle.dump(vectorizer, f)

print(f"✅ Model v{model_version['version']} saved with metadata")
print(f"📁 Model: ../models/{model_filename}")
print(f"📄 Metadata: ../models/{metadata_filename}")
print(f"📄 Vectorizer: ../models/{vectorizer_filename}")
print("\n📊 Metadata:")
print(json.dumps(model_version, indent=2))

## Step 2: Simple Model Registry (Mock)

A **model registry** tracks all model versions and their status (staging, production, archived).

In production, this would be **Azure ML Model Registry** or **MLflow**. Here, we use a simple Python dictionary as a demonstration.

In [ ]:
# Mock model registry
MODEL_REGISTRY = {
    "sentiment_classifier": {
        "versions": {
            "1.0.0": {
                "status": "production",
                "accuracy": model_version["accuracy"],
                "deployed_date": model_version["timestamp"],
                "path": f"../models/{model_filename}"
            },
            "0.9.0": {
                "status": "archived",
                "accuracy": 0.82,
                "deployed_date": "2024-12-01T10:00:00",
                "path": "../models/sentiment_model_v0.9.0.pkl"
            }
        },
        "production_version": "1.0.0"
    }
}

def get_production_model(model_name):
    """Load production model from registry"""
    registry_entry = MODEL_REGISTRY[model_name]
    prod_version = registry_entry["production_version"]
    model_path = registry_entry["versions"][prod_version]["path"]
    
    with open(model_path, 'rb') as f:
        return pickle.load(f), prod_version

def list_model_versions(model_name):
    """List all versions of a model"""
    registry_entry = MODEL_REGISTRY[model_name]
    print(f"\n📦 MODEL REGISTRY: {model_name}")
    print("=" * 70)
    print(f"Production Version: {registry_entry['production_version']}")
    print("\nAll Versions:")
    for version, info in registry_entry["versions"].items():
        status_emoji = "🟢" if info["status"] == "production" else "⚪"
        print(f"  {status_emoji} v{version}: {info['status']} | Accuracy: {info['accuracy']:.2%} | Deployed: {info['deployed_date'][:10]}")
    print("=" * 70)

# Demo: List versions
list_model_versions("sentiment_classifier")

# Demo: Load production model
prod_model, version = get_production_model("sentiment_classifier")
print(f"\n✅ Loaded production model: v{version}")

## Step 3: Model Validation Checks

Before deploying, we run **automated validation checks** to ensure quality.

### Deployment Criteria:
1. ✅ Accuracy >= 80%
2. ✅ All classes have recall >= 70% (balanced performance)
3. ✅ Model size < 50 MB (deployment efficiency)

In [ ]:
import sys
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

def validate_model_for_deployment(model, vectorizer, X_test, y_test, min_accuracy=0.80, min_recall=0.70, max_size_mb=50):
    """
    Validate model against deployment criteria.
    
    Returns:
        bool: True if model passes all checks, False otherwise
    """
    
    checks = {
        "passed": [],
        "failed": []
    }
    
    # Transform test data
    X_test_tfidf = vectorizer.transform(X_test)
    y_pred = model.predict(X_test_tfidf)
    
    # Check 1: Minimum accuracy
    test_accuracy = accuracy_score(y_test, y_pred)
    if test_accuracy >= min_accuracy:
        checks["passed"].append(f"✅ Accuracy: {test_accuracy:.2%} (>= {min_accuracy:.0%})")
    else:
        checks["failed"].append(f"❌ Accuracy: {test_accuracy:.2%} (< {min_accuracy:.0%})")
    
    # Check 2: Balanced performance across classes
    report = classification_report(y_test, y_pred, output_dict=True)
    class_recalls = {label: report[label]['recall'] for label in ['negative', 'neutral', 'positive']}
    min_class_recall = min(class_recalls.values())
    worst_class = [label for label, recall in class_recalls.items() if recall == min_class_recall][0]
    
    if min_class_recall >= min_recall:
        checks["passed"].append(f"✅ Min class recall ({worst_class}): {min_class_recall:.2%} (>= {min_recall:.0%})")
    else:
        checks["failed"].append(f"❌ Min class recall ({worst_class}): {min_class_recall:.2%} (< {min_recall:.0%})")
    
    # Check 3: Model size
    model_size_mb = sys.getsizeof(pickle.dumps(model)) / (1024 * 1024)
    vectorizer_size_mb = sys.getsizeof(pickle.dumps(vectorizer)) / (1024 * 1024)
    total_size_mb = model_size_mb + vectorizer_size_mb
    
    if total_size_mb < max_size_mb:
        checks["passed"].append(f"✅ Total model size: {total_size_mb:.2f} MB (< {max_size_mb} MB)")
    else:
        checks["failed"].append(f"❌ Total model size: {total_size_mb:.2f} MB (>= {max_size_mb} MB)")
    
    # Decision
    can_deploy = len(checks["failed"]) == 0
    
    # Print report
    print("\n🔍 MODEL VALIDATION REPORT")
    print("=" * 70)
    print("\n✅ PASSED CHECKS:")
    for check in checks["passed"]:
        print(f"  {check}")
    
    if checks["failed"]:
        print("\n❌ FAILED CHECKS:")
        for check in checks["failed"]:
            print(f"  {check}")
    
    print("\n" + "=" * 70)
    
    if can_deploy:
        print("🚀 DECISION: Model APPROVED for deployment")
    else:
        print("🛑 DECISION: Model REJECTED - Fix issues before deployment")
    
    print("=" * 70)
    
    return can_deploy

# Run validation
can_deploy = validate_model_for_deployment(model, vectorizer, X_test, y_test)

if can_deploy:
    print("\n✅ Model ready for deployment to production!")
    print("📦 Next step: Deploy via API (see next notebook)")
else:
    print("\n⚠️  Model needs improvement before deployment.")
    print("💡 Suggestions: Collect more data, tune hyperparameters, or try different algorithms.")

## 💡 Key Takeaways: MLOps Basics

### What We Just Did:

1. **Model Versioning** 📦
   - Saved model with version number (v1.0.0)
   - Stored metadata (accuracy, precision, hyperparameters)
   - Can track multiple versions over time

2. **Model Registry** 📋
   - Centralized tracking of all models
   - Know which version is in production
   - Easy rollback if new version fails

3. **Model Validation** ✅
   - Automated checks before deployment
   - Prevent bad models from reaching production
   - Quality gates (accuracy, balance, size)

### In Production (Albert Heijn):

| What We Did | Real Production Tool                    |
|-------------|-----------------------------------------|
| Python dict registry | **Azure Container Registry** or **MLflow** |
| Manual validation | **CI/CD pipeline** (automated)          |
| Local files | **Azure Blob Storage** (centralized)    |

### Why This Matters:

- 🚀 **Faster debugging**: "Which model version is in production?"
- 🔒 **Safety**: Bad models can't deploy automatically
- 📊 **Tracking**: Compare model versions over time
- 🔄 **Rollback**: Revert to previous version if issues arise

---

## 🚀 Next: Deployment Practice

Now that our model is validated and versioned, let's deploy it as an API!

**Next notebook**: `04_deployment_practice.ipynb`